In [1]:
%env CUDA_VISIBLE_DEVICES=4
import logging
from typing import *

import hkkang_utils.file as file_utils
import torch
import tqdm

from colbert.noun_extraction.utils import unidecode_text
from colbert.utils.utils import isint
from model.late_encoder import ColBERTRetriever, RetrievalResult
from model.utils import Document
from scripts.evaluate.utils import get_recall_rates
from scripts.utils import read_collection, read_qrels, read_queries

# Retriever path
ROOT = "/root/ColBERT/experiments/"
EXPERIMENT = "msmarco"
CHECKPOINT_PATH = "/root/ColBERT/checkpoint/baseline_v1/"
# CHECKPOINT_PATH = "/root/ColBERT/checkpoint/ours_v1-4500/"
CORPUS_PATH = "/root/ColBERT/data/msmarco/collection.tsv"
SKIP_PADDING = True
IS_USE_MIN_THRESHOLD = False
IS_USE_PHRASE_LEVEL = False
# Data path
QUERY_PATH = "/root/ColBERT/data/msmarco/queries.dev.tsv"
QRELS_PATH = "/root/ColBERT/data/msmarco/qrels.dev.tsv"
COLLECTION_PATH = "/root/ColBERT/data/msmarco/collection.tsv"
DEV_PATH = "/root/ColBERT/data/msmarco/dev_data.jsonl"


logger = logging.getLogger("TokenScore")

env: CUDA_VISIBLE_DEVICES=4


In [2]:
# initialize retriever
retriever = ColBERTRetriever(root=ROOT, 
                                 checkpoint_path=CHECKPOINT_PATH, corpus_path=CORPUS_PATH,
                                 experiment= EXPERIMENT, skip_padding=SKIP_PADDING, 
                                 is_use_phrase_level=IS_USE_PHRASE_LEVEL, 
                                 show_progress=False, skip_loading=True)

# Tokenizer for query texts
q_tokenizer = retriever.searcher.checkpoint.query_tokenizer
# Tokenizer for documents
d_tokenizer = retriever.searcher.checkpoint.doc_tokenizer
skiplist = list(key for key in retriever.searcher.checkpoint.skiplist.keys() if not isint(key))

[Feb 15, 07:02:33] #> Loading collection...
0M 1M 2M 3M 4M 5M 6M 7M 8M 

In [3]:
# Read in collection
collection: Dict[str, str] = read_collection(COLLECTION_PATH)
collection = {int(k): v for k, v in collection.items()}

In [4]:
# Load queries
sample_num = 100
queries: Dict[str, str] = read_queries(QUERY_PATH)
dev_data: List[List[int]] = file_utils.read_jsonl_file(DEV_PATH)
dev_data = dev_data[:sample_num]

In [5]:
queries: List[str] = [queries[str(qid)] for qid, *pids in dev_data]
gold_pids: List[int] = [pids[0] for qid, *pids in dev_data]
neg_pids: List[List[int]] = [pids[1:] for qid, *pids in dev_data]

In [6]:
# Retrieve top-100 passages for each query
all_results: List[List[RetrievalResult]] = []
for i in tqdm.tqdm(range(len(queries))):
    query = queries[i]
    pids = [gold_pids[i]] + neg_pids[i]
    results: List[RetrievalResult] = retriever.calculate_score_batch(queries=[query]*len(pids), pids=pids)
    # Append pids to the result
    assert len(results) == len(pids), f"Length of result and pids are different: {len(results)} vs {len(pids)}"
    for j in range(len(results)):
        results[j].doc.id = pids[j]
    # Sort the results by their scores
    results = sorted(results, key=lambda x: x.score, reverse=True)
    all_results.append(results)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:24<00:00,  4.10it/s]


In [7]:
# Evaluate the ranking results
dicts = {}
for i in range(len(queries)):
    rank_results: List[RetrievalResult] = all_results[i]
    gold_pid: int = gold_pids[i]
    ranked_pids: List[int] = [rank_result.doc.id for rank_result in rank_results]
    recall = get_recall_rates(ranked_pids=ranked_pids, gold_pids=[gold_pid])
    for key, value in recall.items():
        if key not in dicts:
            dicts[key] = []
        dicts[key].append(value)
# Average the recall rates
for key, value in dicts.items():
    print(f"{key}: {sum(value)/len(value)}")

@5: 0.48
@10: 0.57
@50: 0.81
@100: 0.99


In [8]:
def get_ranks(scores: List[float]) -> List[int]:
    ranks = []
    for i in range(len(scores)):
        rank = 1
        for j in range(len(scores)):
            if scores[j] > scores[i]:
                rank += 1
        ranks.append(rank)
    return ranks

In [9]:

def examine_token_scores(q_toks: List[str], d_toks: List[str], tok_scores: List[List[float]]) -> None:
    tok_scores = torch.tensor(tok_scores)
    # Find the max score
    max_scores = tok_scores.max(dim=1)[0]
    # Show max scores
    for q_i in range(len(q_toks)):
        if q_toks[q_i] == "[MASK]" and max_scores[q_i] == 0.0:
            continue
        print(f"Query token {q_i}: {q_toks[q_i]} (max score: {max_scores[q_i]})")
    print("\n")
    for q_i in range(len(q_toks)):
        if q_toks[q_i] == "[MASK]" and max_scores[q_i] == 0.0:
            continue
        print(f"Query token {q_i}: {q_toks[q_i]} (max score: {max_scores[q_i]})")
        # Rank the document tokens by score
        ranks = get_ranks(tok_scores[q_i])
        for d_i in range(len(d_toks)):
            print(f"\tDocument token {d_i}: {d_toks[d_i]} (score: {tok_scores[q_i][d_i]}, rank: {ranks[d_i]})")
        print("\n")
    return None

In [10]:
# Show gold and non-gold passages
def show_passages(query: str, docs: List[Document], gold_pid: int, scores: List[int]) -> None:
    print("Query: ", query)
    print(f"Gold document: (PID: {gold_pid})")
    print(f"{unidecode_text(collection[gold_pid])}\n")
    # Figure out if the gold passage is in the top-100
    gold_idx = -1
    for i, doc in enumerate(docs):
        if doc.id == gold_pid:
            gold_idx = i
            break
    gold_in_top_100 = True if gold_idx != -1 else False
    # Print document texts
    print(f"Gold passage ranked: {gold_idx}")
    for i in range(max(10, gold_in_top_100)):
        doc = docs[i]
        score = scores[i]
        print(f"Document {i+1}: {doc.title} (PID: {doc.id}) (score: {score})")
        print(f"{doc.text}\n")

In [11]:
def examine(query: str, pid: int) -> None:
    doc = unidecode_text(collection[pid])

    # Tokenize
    q_toks = q_tokenizer.tokenize([query], add_special_tokens=True)[0]
    d_toks = d_tokenizer.tokenize([doc], add_special_tokens=True)[0]
    if IS_USE_PHRASE_LEVEL:
        # Get phrase indices
        q_phrase_indices = retriever.searcher.checkpoint.get_phrase_indices(texts=[query], tok_ids=None, masks=None, full_length_search=False, is_query=True)[0][0]
        d_phrase_indices = retriever.searcher.checkpoint.get_phrase_indices(texts=[doc], tok_ids=None, masks=None, full_length_search=False, is_query=False)[0][0]
        # Combine tokens into phrases
        q_toks = [" ".join(q_toks[s:e]) for s, e in q_phrase_indices]
        d_toks = [" ".join(d_toks[s:e]) for s, e in d_phrase_indices]
        
    # # Remove doc toks in skiplists
    # d_toks = [d_tok for d_tok in d_toks if d_tok not in skiplist]    

    # Compute token scores
    result: RetrievalResult = retriever.calculate_score_by_text_batch(queries=[query], doc_texts=[doc])[0]

    print(f"Query: {query}")
    print(f"Document (score: {result.score}):\n{doc}\n")
    examine_token_scores(q_toks, d_toks, result.token_scores)

In [15]:
# Find the wrong indices
wrong_indices = []
for idx in range(len(all_results)):
    pred_pids = [result.doc.id for result in all_results[idx]]
    if gold_pids[idx] not in pred_pids[:10]:
        wrong_indices.append(idx)
print(wrong_indices)

[0, 4, 7, 12, 13, 14, 16, 17, 18, 20, 22, 24, 25, 26, 28, 29, 32, 33, 34, 35, 37, 38, 42, 45, 50, 51, 58, 60, 62, 64, 65, 68, 69, 71, 75, 76, 79, 84, 87, 89, 93, 94, 96]


In [16]:
idx = 2
query = queries[idx]
gold_pid = gold_pids[idx]
pred_docs = [result.doc for result in all_results[idx]]
pred_scores = [result.score for result in all_results[idx]]
pred_pids = [doc.id for doc in pred_docs]
gold_found = gold_pid in pred_pids
print(f"Is in the gold?: {gold_found}, at {pred_pids.index(gold_pid) if gold_found else -1}")
show_passages(query=query, docs=pred_docs, gold_pid=gold_pid, scores=pred_scores)

Is in the gold?: True, at 7
Query:  symptoms of a dying mouse
Gold document: (PID: 7066900)
The symptoms are similar but the mouse will be in much worse condition: runny eyes; sneezing; wheezing; shaking; fluctuating body temperature; tiredness; loss of appetite; dull coat; If these symptoms persist or worsen, or the mouse becomes limp and struggles to walk, then the mouse must go to the vet's or it could die. To prevent influenza, do not touch your pet if you have flu, as mice catch it from humans.

Gold passage ranked: 7
Document 1:  (PID: 6623912) (score: 27.79988670349121)
Expert: Theresa replied 8 years ago. Symptoms of a dying dog include, lack of appetite, pale, blue, or gray gums, dry gums, inability to stand, dilated pupils, difficulty breathing, unresponsiveness, and neurologic symptoms. A dog can live for a couple weeks without enough food and water.

Document 2:  (PID: 7066895) (score: 27.49756622314453)
This can be fatal quite quickly to mice. 1  It is caused by dusty sawd

In [17]:
# HERE: examine the gold passages
idx = 2
query = queries[idx]
gold_pid = gold_pids[idx]
examine(query=query, pid=gold_pid)

Query: symptoms of a dying mouse
Document (score: 26.868736267089844):
The symptoms are similar but the mouse will be in much worse condition: runny eyes; sneezing; wheezing; shaking; fluctuating body temperature; tiredness; loss of appetite; dull coat; If these symptoms persist or worsen, or the mouse becomes limp and struggles to walk, then the mouse must go to the vet's or it could die. To prevent influenza, do not touch your pet if you have flu, as mice catch it from humans.

Query token 0: [CLS] (max score: 3.67244029045105)
Query token 1: [Q] (max score: 3.5361099243164062)
Query token 2: symptoms (max score: 3.662522792816162)
Query token 3: of (max score: 3.6285717487335205)
Query token 4: a (max score: 3.558450937271118)
Query token 5: dying (max score: 1.9699530601501465)
Query token 6: mouse (max score: 3.6906113624572754)
Query token 7: [SEP] (max score: 3.1500749588012695)


Query token 0: [CLS] (max score: 3.67244029045105)
	Document token 0: [CLS] (score: 3.6602623462677

In [18]:
# HERE: examine the non-gold passages
pid = 6623912
examine(query=query, pid=pid)

Query: symptoms of a dying mouse
Document (score: 27.799901962280273):
Expert: Theresa replied 8 years ago. Symptoms of a dying dog include, lack of appetite, pale, blue, or gray gums, dry gums, inability to stand, dilated pupils, difficulty breathing, unresponsiveness, and neurologic symptoms. A dog can live for a couple weeks without enough food and water.

Query token 0: [CLS] (max score: 3.6873679161071777)
Query token 1: [Q] (max score: 3.723837375640869)
Query token 2: symptoms (max score: 3.8135530948638916)
Query token 3: of (max score: 3.728806972503662)
Query token 4: a (max score: 3.6380677223205566)
Query token 5: dying (max score: 3.5109267234802246)
Query token 6: mouse (max score: 2.457564353942871)
Query token 7: [SEP] (max score: 3.239777088165283)


Query token 0: [CLS] (max score: 3.6873679161071777)
	Document token 0: [CLS] (score: 3.587080478668213, rank: 3)
	Document token 1: [D] (score: 2.6562981605529785, rank: 21)
	Document token 2: expert (score: 0.60952413082